In [1]:
import pandas as pd

import json
import os
import ast

BASE_DIR = 'Datasets/DDXPlus'

print(f'Current Directory: {os.getcwd()}')
print(f'Data Directory: {BASE_DIR}')

required_files = ['train.csv', 'release_evidences.json', 'release_conditions.json']
for f in required_files:
    path = os.path.join(BASE_DIR, f)
    if os.path .exists(path):
        print(f'Found file: {f}')
    else:
        print(f'File not found: {f}')

Current Directory: /Users/yufan.shi/Desktop/ECE285 Project
Data Directory: Datasets/DDXPlus
Found file: train.csv
Found file: release_evidences.json
Found file: release_conditions.json


In [2]:
evidence_path = os.path.join(BASE_DIR, 'release_evidences.json')
with open(evidence_path, 'r') as f:
    evidence_dict_raw = json.load(f)

condition_path = os.path.join(BASE_DIR, 'release_conditions.json')
with open(condition_path, 'r') as f:
    condition_dict_raw = json.load(f)

print(f"Total symptoms: {len(evidence_dict_raw)}")
print(f"Total conditions: {len(condition_dict_raw)}")

print("\n---Evidences---")
first_ev_key = list(evidence_dict_raw.keys())[0]
print(f"Key: {first_ev_key}")
print(f"Value: {json.dumps(evidence_dict_raw[first_ev_key], indent=2, ensure_ascii=False)}")

print("\n---Conditions---")
first_cond_key = list(condition_dict_raw.keys())[0]
print(f"Key: {first_cond_key}")
print(f"Value: {json.dumps(condition_dict_raw[first_cond_key], indent=2, ensure_ascii=False)}")

Total symptoms: 223
Total conditions: 49

---Evidences---
Key: E_91
Value: {
  "name": "E_91",
  "code_question": "E_91",
  "question_fr": "Avez-vous objectivé ou ressenti de la fièvre?",
  "question_en": "Do you have a fever (either felt or measured with a thermometer)?",
  "is_antecedent": false,
  "default_value": 0,
  "value_meaning": {},
  "possible-values": [],
  "data_type": "B"
}

---Conditions---
Key: Spontaneous pneumothorax
Value: {
  "condition_name": "Spontaneous pneumothorax",
  "cond-name-fr": "Pneumothorax spontané",
  "cond-name-eng": "Spontaneous pneumothorax",
  "icd10-id": "J93",
  "symptoms": {
    "E_55": {},
    "E_53": {},
    "E_57": {},
    "E_54": {},
    "E_59": {},
    "E_56": {},
    "E_58": {},
    "E_66": {},
    "E_220": {},
    "E_218": {},
    "E_14": {},
    "E_151": {}
  },
  "antecedents": {
    "E_79": {},
    "E_165": {},
    "E_21": {},
    "E_123": {},
    "E_204": {}
  },
  "severity": 2
}


In [3]:
train_path = os.path.join(BASE_DIR, "train.csv")
train_df = pd.read_csv(train_path)

print("train shape:", train_df.shape)
print("\nColumns (first 80):")
print(list(train_df.columns)[:80])

display(train_df.head(3))

train shape: (1025602, 6)

Columns (first 80):
['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']


,AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
0,18,"[['Bronchitis', 0.19171203430383882], ['Pneumo...",M,URTI,"['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54...",E_91
1,21,"[['HIV (initial infection)', 0.518950056440760...",M,HIV (initial infection),"['E_9', 'E_27', 'E_50', 'E_51', 'E_53', 'E_54_...",E_50
2,19,"[['Bronchitis', 0.11278064619119596], ['Pneumo...",F,Pneumonia,"['E_53', 'E_54_@_V_179', 'E_54_@_V_192', 'E_55...",E_77


In [4]:
def detect_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for k in candidates:
        if k.lower() in cols_lower:
            return cols_lower[k.lower()]
    return None

label_col = detect_col(train_df, ["PATHOLOGY", "DISEASE", "CONDITION", "LABEL", "target"])
age_col   = detect_col(train_df, ["AGE", "age"])
sex_col   = detect_col(train_df, ["SEX", "sex", "GENDER", "gender"])
evid_col  = detect_col(train_df, ["EVIDENCES", "EVIDENCE", "symptoms", "features"])

print("label_col:", label_col)
print("age_col  :", age_col)
print("sex_col  :", sex_col)
print("evid_col :", evid_col)

label_col: PATHOLOGY
age_col  : AGE
sex_col  : SEX
evid_col : EVIDENCES


In [5]:
raw_e = train_df.loc[0, "EVIDENCES"]
raw_i = train_df.loc[0, "INITIAL_EVIDENCE"]
raw_ddx = train_df.loc[0, "DIFFERENTIAL_DIAGNOSIS"]

print("RAW INITIAL_EVIDENCE:", raw_i)
print("RAW EVIDENCES (first 300 chars):", str(raw_e)[:300])
print("RAW DIFFERENTIAL_DIAGNOSIS (first 300 chars):", str(raw_ddx)[:300])

RAW INITIAL_EVIDENCE: E_91
RAW EVIDENCES (first 300 chars): ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']
RAW DIFFERENTIAL_DIAGNOSIS (first 300 chars): [['Bronchitis', 0.19171203430383882], ['Pneumonia', 0.17579340398940366], ['URTI', 0.1607809719801254], ['Bronchiectasis', 0.12429044460990353], ['Tuberculosis', 0.11367177304035844], ['Influenza', 0.11057936110639896], ['HIV (initial infection)', 0.07333003867293564], ['Chagas', 0.04984197229703562


In [6]:
def parse_list_cell(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return []
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return [t.strip() for t in s.replace(",", ";").split(";") if t.strip()]

if evid_col is not None:
    tokens = parse_list_cell(train_df.loc[0, evid_col])
    print("Evidence tokens (row0):", tokens[:30])
    print("Total tokens:", len(tokens))

Evidence tokens (row0): ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']
Total tokens: 19


In [9]:
import re

def parse_ddxplus_token(tok: str):
    """
    支持两种：
    1) 'E_91' -> (code='E_91', has_value=False, value=None)
    2) 'E_54_@_V_161' or 'E_56_@_4' -> (code='E_54', has_value=True, value='V_161' or '4')
    """
    t = str(tok).strip().strip('"').strip("'")
    if not t:
        return None

    if "_@_" in t:
        # 只按第一个 _@_ 切
        code, val = t.split("_@_", 1)
        return code.strip(), True, val.strip()

    return t, False, None

# 用你 row0 的 tokens 测试
tokens0 = parse_list_cell(train_df.loc[0, "EVIDENCES"])

parsed = [parse_ddxplus_token(t) for t in tokens0] # type: ignore
print("First 15 parsed:")
for x in parsed[:15]:
    print(x)

# 简单统计：有多少带 value
with_value = sum(1 for x in parsed if x and x[1])
print("\nTotal tokens:", len(parsed))
print("Tokens with value:", with_value)


First 15 parsed:
('E_48', False, None)
('E_50', False, None)
('E_53', False, None)
('E_54', True, 'V_161')
('E_54', True, 'V_183')
('E_55', True, 'V_89')
('E_55', True, 'V_108')
('E_55', True, 'V_167')
('E_56', True, '4')
('E_57', True, 'V_123')
('E_58', True, '3')
('E_59', True, '3')
('E_77', False, None)
('E_79', False, None)
('E_91', False, None)

Total tokens: 19
Tokens with value: 10


In [10]:
def normalize_tokens_with_meta(tokens, evidence_dict):
    out = []
    for tok in tokens:
        parsed = parse_ddxplus_token(tok)
        if parsed is None:
            continue
        code, has_value, val = parsed

        meta = evidence_dict.get(code, {})
        text = meta.get("question_en") or meta.get("name") or code
        dt   = meta.get("data_type")
        vm   = meta.get("value_meaning", {}) or {}

        ev = {
            "code": code,
            "text": text,
            "data_type": dt,
            "state": "value" if has_value else "present"
        }

        if has_value:
            v_raw = val
            v_num = None
            try:
                v_num = float(v_raw) if "." in str(v_raw) else int(v_raw) # type: ignore
            except Exception:
                v_num = None

            ev["value"] = v_num if v_num is not None else v_raw

            meaning = None
            for k in [str(v_raw), str(ev["value"]), str(v_raw).replace("V_", ""), str(ev["value"]).replace("V_", "")]:
                if k in vm:
                    meaning = vm[k]
                    break
            if meaning not in (None, "", {}):
                ev["meaning"] = meaning

        out.append(ev)

    return out


tokens0 = parse_list_cell(train_df.loc[0, "EVIDENCES"])
norm_evs = normalize_tokens_with_meta(tokens0, evidence_dict_raw)

print("Total normalized evidences:", len(norm_evs))
print("With meaning:", sum(1 for e in norm_evs if "meaning" in e))
print("\nFirst 10 normalized evidences:")
norm_evs[:10]


Total normalized evidences: 19
With meaning: 7

First 10 normalized evidences:


[{'code': 'E_48',
  'text': 'Do you live with 4 or more people?',
  'data_type': 'B',
  'state': 'present'},
 {'code': 'E_50',
  'text': 'Have you had significantly increased sweating?',
  'data_type': 'B',
  'state': 'present'},
 {'code': 'E_53',
  'text': 'Do you have pain somewhere, related to your reason for consulting?',
  'data_type': 'B',
  'state': 'present'},
 {'code': 'E_54',
  'text': 'Characterize your pain:',
  'data_type': 'M',
  'state': 'value',
  'value': 'V_161',
  'meaning': {'fr': 'sensible', 'en': 'sensitive'}},
 {'code': 'E_54',
  'text': 'Characterize your pain:',
  'data_type': 'M',
  'state': 'value',
  'value': 'V_183',
  'meaning': {'fr': 'une lourdeur', 'en': 'heavy'}},
 {'code': 'E_55',
  'text': 'Do you feel pain somewhere?',
  'data_type': 'M',
  'state': 'value',
  'value': 'V_89',
  'meaning': {'fr': 'front', 'en': 'forehead'}},
 {'code': 'E_55',
  'text': 'Do you feel pain somewhere?',
  'data_type': 'M',
  'state': 'value',
  'value': 'V_108',
  'mean

In [11]:
def compact_evidence_list(evs):
    """
    输入: norm_evs (list of dict)
    输出: 合并同 code 的 evidence，适合写入最终 record/JSONL
    """
    merged = {}
    order = []

    for e in evs:
        code = e["code"]
        if code not in merged:
            merged[code] = {
                "code": code,
                "text": e.get("text", code),
                "data_type": e.get("data_type"),
                "present": False,
                "values": []   # 多选/数值都放这里
            }
            order.append(code)

        if e.get("state") == "present":
            merged[code]["present"] = True
        else:
            v = e.get("value")
            meaning = e.get("meaning", None)

            meaning_en = None
            if isinstance(meaning, dict):
                meaning_en = meaning.get("en")
            elif isinstance(meaning, str):
                meaning_en = meaning

            item = {"value": v}
            if meaning_en not in (None, ""):
                item["meaning_en"] = meaning_en

            merged[code]["values"].append(item)

    return [merged[c] for c in order]


compact = compact_evidence_list(norm_evs)

print("Original evidences:", len(norm_evs))
print("Compacted evidences:", len(compact))
print("\nFirst 5 compacted:")
compact[:5]


Original evidences: 19
Compacted evidences: 16

First 5 compacted:


[{'code': 'E_48',
  'text': 'Do you live with 4 or more people?',
  'data_type': 'B',
  'present': True,
  'values': []},
 {'code': 'E_50',
  'text': 'Have you had significantly increased sweating?',
  'data_type': 'B',
  'present': True,
  'values': []},
 {'code': 'E_53',
  'text': 'Do you have pain somewhere, related to your reason for consulting?',
  'data_type': 'B',
  'present': True,
  'values': []},
 {'code': 'E_54',
  'text': 'Characterize your pain:',
  'data_type': 'M',
  'present': False,
  'values': [{'value': 'V_161', 'meaning_en': 'sensitive'},
   {'value': 'V_183', 'meaning_en': 'heavy'}]},
 {'code': 'E_55',
  'text': 'Do you feel pain somewhere?',
  'data_type': 'M',
  'present': False,
  'values': [{'value': 'V_89', 'meaning_en': 'forehead'},
   {'value': 'V_108', 'meaning_en': 'cheek(R)'},
   {'value': 'V_167', 'meaning_en': 'temple(L)'}]}]

In [24]:
def normalize_ddx_list(ddx_raw):
    """
    输入：[['Bronchitis', 0.19], ['Pneumonia', 0.17], ...]
    输出：[{name:..., prob:...}, ...]
    """
    ddx = []
    if isinstance(ddx_raw, list):
        for item in ddx_raw:
            if isinstance(item, list) and len(item) >= 2:
                name = str(item[0]).strip()
                try:
                    prob = float(item[1])
                except Exception:
                    prob = None
                ddx.append({"name": name, "prob": prob})
            elif isinstance(item, str):
                ddx.append({"name": item.strip(), "prob": None})
    return ddx


def normalize_initial_evidence(init_tok):
    """
    把 INITIAL_EVIDENCE（如 'E_91' 或 'E_56_@_4'）也标准化
    """
    parsed = parse_ddxplus_token(init_tok)
    if parsed is None:
        return None
    code, has_value, val = parsed
    meta = evidence_dict_raw.get(code, {})
    ev = {
        "code": code,
        "text": meta.get("question_en") or meta.get("name") or code,
        "data_type": meta.get("data_type"),
        "state": "value" if has_value else "present"
    }
    if has_value:
        ev["value"] = val
        vm = meta.get("value_meaning", {}) or {}
        if val in vm and isinstance(vm[val], dict) and "en" in vm[val]:
            ev["meaning_en"] = vm[val]["en"]
    return ev


def build_record_from_row(df, idx, split="train"):
    row = df.loc[idx]

    # label
    pathology = str(row["PATHOLOGY"]).strip()
    cond_meta = condition_dict_raw.get(pathology, {})

    # evidences -> norm -> compact
    tokens = parse_list_cell(row["EVIDENCES"])
    norm_evs = normalize_tokens_with_meta(tokens, evidence_dict_raw)  # 你之前那版函数
    compact_evs = compact_evidence_list(norm_evs)                     # 你之前那版函数

    # initial evidence
    init_ev = normalize_initial_evidence(row["INITIAL_EVIDENCE"])

    # ddx list
    ddx_raw = parse_list_cell(row["DIFFERENTIAL_DIAGNOSIS"])
    ddx = normalize_ddx_list(ddx_raw)

    rec = {
        "split": split,
        "case_id": f"{split}_{idx}",
        "age": int(float(row["AGE"])) if not pd.isna(row["AGE"]) else None,
        "sex": str(row["SEX"]).strip() if not pd.isna(row["SEX"]) else None,

        "initial_evidence": init_ev,
        "evidence": compact_evs,

        "label": {
            "name": pathology,
            "icd10": cond_meta.get("icd10-id"),
            "severity": cond_meta.get("severity")
        },

        "differential_diagnosis": ddx
    }
    return rec


record0 = build_record_from_row(train_df, 0, split="train")

print("Label:", record0["label"])
print("Initial evidence:", record0["initial_evidence"])
print("Evidence (first 3):", record0["evidence"][:3])
print("DDX (first 5):", record0["differential_diagnosis"][:5])


Label: {'name': 'URTI', 'icd10': 'j06.9', 'severity': 5}
Initial evidence: {'code': 'E_91', 'text': 'Do you have a fever (either felt or measured with a thermometer)?', 'data_type': 'B', 'state': 'present'}
Evidence (first 3): [{'code': 'E_48', 'text': 'Do you live with 4 or more people?', 'data_type': 'B', 'present': True, 'values': []}, {'code': 'E_50', 'text': 'Have you had significantly increased sweating?', 'data_type': 'B', 'present': True, 'values': []}, {'code': 'E_53', 'text': 'Do you have pain somewhere, related to your reason for consulting?', 'data_type': 'B', 'present': True, 'values': []}]
DDX (first 5): [{'name': 'Bronchitis', 'prob': 0.19171203430383882}, {'name': 'Pneumonia', 'prob': 0.17579340398940366}, {'name': 'URTI', 'prob': 0.1607809719801254}, {'name': 'Bronchiectasis', 'prob': 0.12429044460990353}, {'name': 'Tuberculosis', 'prob': 0.11367177304035844}]


In [28]:
def normalize_yesno(s):
    """把各种写法统一成 YES/NO；不是 yes/no 的返回 None"""
    if s is None:
        return None
    t = str(s).strip().lower()
    yes_set = {"y", "yes", "true", "1", "oui"}
    no_set  = {"n", "no", "false", "0", "non"}
    if t in yes_set:
        return "YES"
    if t in no_set:
        return "NO"
    return None


def compact_evidence_to_text_v2(compact_evs, max_lines=None):
    lines = []
    for e in compact_evs:
        text = e.get("text", e.get("code"))
        dt = e.get("data_type")
        present = e.get("present", False)
        values = e.get("values", [])

        # B型：present=True 才输出 YES
        if dt == "B":
            if present:
                lines.append(f"- {text} YES")
            continue

        # 非B型：有 values 才输出
        if values:
            rendered_vals = []
            for item in values:
                # 优先 meaning_en
                meaning_en = item.get("meaning_en") if isinstance(item, dict) else None
                v = item.get("value") if isinstance(item, dict) else item

                yn = normalize_yesno(meaning_en)
                if yn is not None:
                    rendered_vals.append(yn)
                else:
                    rendered_vals.append(str(meaning_en) if meaning_en not in (None, "") else str(v))

            # 去重保持顺序
            seen = set()
            uniq = []
            for v in rendered_vals:
                if v not in seen:
                    uniq.append(v)
                    seen.add(v)

            # 如果这一题其实是 Yes/No（比如只有 YES 或只有 NO），就只输出一个
            if len(uniq) == 1 and uniq[0] in {"YES", "NO"}:
                lines.append(f"- {text} {uniq[0]}")
            else:
                lines.append(f"- {text} " + "; ".join(uniq))

        else:
            # 没有 values 但 present=True 的情况（少见）也允许输出
            if present:
                lines.append(f"- {text} YES")

        if max_lines is not None and len(lines) >= max_lines:
            break

    return "\n".join(lines)


# 用 v2 重建 prompt_text
evidence_text_v2 = compact_evidence_to_text_v2(record0["evidence"])
print(evidence_text_v2)


- Do you live with 4 or more people? YES
- Have you had significantly increased sweating? YES
- Do you have pain somewhere, related to your reason for consulting? YES
- Characterize your pain: sensitive; heavy
- Do you feel pain somewhere? forehead; cheek(R); temple(L)
- How intense is the pain? 4
- Does the pain radiate to another location? nowhere
- How precisely is the pain located? 3
- How fast did the pain appear? 3
- Do you have a cough that produces colored or more abundant sputum than usual? YES
- Do you smoke cigarettes? YES
- Do you have a fever (either felt or measured with a thermometer)? YES
- Do you have a sore throat? YES
- Do you have a cough? YES
- Have you traveled out of the country in the last 4 weeks? NO
- Are you exposed to secondhand cigarette smoke on a daily basis? YES


In [29]:
def compact_evidence_to_text_split(compact_evs, evidence_dict_raw, max_lines=None):
    def normalize_yesno(s):
        if s is None:
            return None
        t = str(s).strip().lower()
        if t in {"y","yes","true","1"}: return "YES"
        if t in {"n","no","false","0"}: return "NO"
        return None

    def render_one(e):
        text = e.get("text", e.get("code"))
        dt = e.get("data_type")
        present = e.get("present", False)
        values = e.get("values", [])

        if dt == "B":
            return f"- {text} YES" if present else None

        if values:
            rendered = []
            for item in values:
                meaning_en = item.get("meaning_en") if isinstance(item, dict) else None
                v = item.get("value") if isinstance(item, dict) else item
                yn = normalize_yesno(meaning_en)
                rendered.append(yn if yn is not None else (meaning_en if meaning_en not in (None,"") else str(v)))

            # 去重保序
            seen, uniq = set(), []
            for r in rendered:
                if r not in seen:
                    uniq.append(r); seen.add(r)

            if len(uniq) == 1 and uniq[0] in {"YES","NO"}:
                return f"- {text} {uniq[0]}"
            return f"- {text} " + "; ".join(uniq)

        if present:
            return f"- {text} YES"
        return None

    symptoms_lines = []
    antecedent_lines = []

    for e in compact_evs:
        code = e.get("code")
        is_ant = bool(evidence_dict_raw.get(code, {}).get("is_antecedent", False))

        line = render_one(e)
        if line is None:
            continue

        if is_ant:
            antecedent_lines.append(line)
        else:
            symptoms_lines.append(line)

        if max_lines is not None and (len(symptoms_lines) + len(antecedent_lines)) >= max_lines:
            break

    out = []
    if symptoms_lines:
        out.append("Symptoms / Current findings:\n" + "\n".join(symptoms_lines))
    if antecedent_lines:
        out.append("Antecedents / Risk factors:\n" + "\n".join(antecedent_lines))
    return "\n\n".join(out)


# 用 record0 测试
print(compact_evidence_to_text_split(record0["evidence"], evidence_dict_raw))


Symptoms / Current findings:
- Have you had significantly increased sweating? YES
- Do you have pain somewhere, related to your reason for consulting? YES
- Characterize your pain: sensitive; heavy
- Do you feel pain somewhere? forehead; cheek(R); temple(L)
- How intense is the pain? 4
- Does the pain radiate to another location? nowhere
- How precisely is the pain located? 3
- How fast did the pain appear? 3
- Do you have a cough that produces colored or more abundant sputum than usual? YES
- Do you have a fever (either felt or measured with a thermometer)? YES
- Do you have a sore throat? YES
- Do you have a cough? YES

Antecedents / Risk factors:
- Do you live with 4 or more people? YES
- Do you smoke cigarettes? YES
- Have you traveled out of the country in the last 4 weeks? NO
- Are you exposed to secondhand cigarette smoke on a daily basis? YES


In [30]:
def make_input_text(record, evidence_dict_raw, split_sections=True):
    age = record.get("age")
    sex = record.get("sex")
    header = [f"Patient: age={age}, sex={sex}"]

    if record.get("initial_evidence") is not None:
        header.append(f"Initial complaint: {record['initial_evidence'].get('text')} (YES)")

    # 这里用你前面做好的“分段 flatten”逻辑（如果你已经保存成函数就直接用）
    if split_sections:
        findings = compact_evidence_to_text_split(record["evidence"], evidence_dict_raw)
    else:
        findings = compact_evidence_to_text_v2(record["evidence"])

    return "\n".join(header) + "\n\n" + findings


def make_sft_example(record, evidence_dict_raw):
    instruction = (
        "You are a clinical decision support model. "
        "Given the patient information and findings, output the single most likely diagnosis "
        "(one disease label only, no explanation)."
    )
    return {
        "instruction": instruction,
        "input": make_input_text(record, evidence_dict_raw, split_sections=True),
        "output": record["label"]["name"]   # 方案A：标准label（URTI等）
    }

# 测试：用你已有的 record0
sft0 = make_sft_example(record0, evidence_dict_raw)
print("OUTPUT:", sft0["output"])
print(sft0["input"][:1200])


OUTPUT: URTI
Patient: age=18, sex=M
Initial complaint: Do you have a fever (either felt or measured with a thermometer)? (YES)

Symptoms / Current findings:
- Have you had significantly increased sweating? YES
- Do you have pain somewhere, related to your reason for consulting? YES
- Characterize your pain: sensitive; heavy
- Do you feel pain somewhere? forehead; cheek(R); temple(L)
- How intense is the pain? 4
- Does the pain radiate to another location? nowhere
- How precisely is the pain located? 3
- How fast did the pain appear? 3
- Do you have a cough that produces colored or more abundant sputum than usual? YES
- Do you have a fever (either felt or measured with a thermometer)? YES
- Do you have a sore throat? YES
- Do you have a cough? YES

Antecedents / Risk factors:
- Do you live with 4 or more people? YES
- Do you smoke cigarettes? YES
- Have you traveled out of the country in the last 4 weeks? NO
- Are you exposed to secondhand cigarette smoke on a daily basis? YES
